#### Simple Chatbot Without LangChain using Groq api

In [ ]:
import os
from dotenv import load_dotenv
from groq import Groq

# Load environment variables from .env file
load_dotenv()

client = Groq()
completion = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
      {
        "role": "user",
        "content": "Hello! I am alok"
      }
    ],
    temperature=1,
    max_completion_tokens=2048,
    top_p=1,
    reasoning_effort="medium",
    stream=True,
    stop=None
)

for chunk in completion:
    print(chunk.choices[0].delta.content or "", end="")




#### Simple Q&A bot with LangChain wiht Groq

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b")

response = llm.invoke("Hello, How are you ?")

print(response.content)

#### Common LLM calls with - `init_chat_model`

In [ ]:
from langchain.chat_models import init_chat_model

model = "groq:openai/gpt-oss-120b"
# "openai:gpt-6-astra" - Using init_chat_model we can switch to any model instantly

llm = init_chat_model(model) 

response = llm.invoke("Hello")
print(response.content)

In [ ]:
from langchain.chat_models import init_chat_model

model = "openai/gpt-oss-120b"

llm = init_chat_model(model, model_provider="groq") 

response = llm.invoke("Hello, give the angry emoji")
print(response.content)

In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

MODEL = os.getenv("MODEL", "openai/gpt-oss-120b")
PROVIDER = os.getenv("PROVIDER", "groq")

llm = init_chat_model(MODEL, model_provider=PROVIDER) 

response = llm.invoke("Hello, give the angry emoji")
print(response.content)


#### Code components of `LangChain`

##### 1. Messages

#### Prompt -> LLM -> 3 Types -> Role of prompt ["User", "ai", "system"]

In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

MODEL = os.getenv("MODEL", "openai/gpt-oss-120b")
PROVIDER = os.getenv("PROVIDER", "groq")

llm = init_chat_model(MODEL, model_provider=PROVIDER) 

res = llm.invoke([
    {"role": "system", "content": "You are a python developr having 7 years of experience in python ,but don't have any idea about java."} ,
    {"role": "user", "content": "How to find the 2nd laregest number in array in java"} ,
])

print(res.content)


In [ ]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

prompts = [
    SystemMessage(content="You are a python developr having 7 years of experience in python ,but don't have any idea about java not even a single thing and you can't answer anything related to java programming."),
    HumanMessage(content="How to find the 2nd laregest number in array in java")
]

res = llm.invoke(prompts)

print(res.content)

##### 2. Prompt Templates

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    {"role":"system", "content":"You are a 7 years of {language} developer"}, 
    {"role":"user", "content":"{query}"}, 
])

In [ ]:
prompt

In [ ]:
prompt = prompt.invoke({"language": "javascript", "query": "How to find the 2nd laregest number in array"})
prompt

#### 3. LCLE - `|` Operator

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# 1. Define the prompt template with your placeholders
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful programming assistant specialized in {language}."),
    ("human", "{query}")
])

# 2. Supply the variables
question = "How to map the array"
language = "javascript"

# 3. Format and pass to the model
finalprompt = prompt_template.invoke({"language": language, "query": question})

res = llm.invoke(finalprompt)
print(res.content)


#### `LangChain` Chain

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Ensure you have the ChatPromptTemplate object
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Reply in {language}."),
    ("human", "{query}")
])

qna_chain = prompt | llm 

res = qna_chain.invoke({"language": "english", "query": "what is gen-ai"})
print(res.content)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Ensure you have the ChatPromptTemplate object
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Reply in {language}."),
    ("human", "{query}")
])

# 2. Custom helper functions
def getContent(res):
    return res.content


qna_chain = prompt_template | llm | getContent

# 4. Invoke the chain
res = qna_chain.invoke({"language": "english", "query": "what is gen-ai"})
print(res)


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.messages import AIMessage

# 1. Ensure you have the ChatPromptTemplate object
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Reply in {language}."),
    ("human", "{query}")
])

# 2. Custom helper functions
def getContent(res: AIMessage):
    return res.content


qna_chain = prompt_template | llm | getContent

# 4. Invoke the chain
res = qna_chain.invoke({"language": "english", "query": "what is gen-ai"})
print(res)


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Ensure you have the ChatPromptTemplate object
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Reply in {language}."),
    ("human", "{query}")
])

# 2. Custom helper functions
def getContent(res):
    return res.content

def shortTheMessage(msg: str):
    return msg[:100]

# 3. Build the chain with prompt_template (NOT prompt)
qna_chain = prompt_template | llm | StrOutputParser() | shortTheMessage

# 4. Invoke the chain
res = qna_chain.invoke({"language": "english", "query": "what is gen-ai"})
print(res)


#### 4. `Memory` - Conversation History

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-20b")

In [ ]:
res = llm.invoke("Who is Pm of india ?")
print(res.content)

In [ ]:
res = llm.invoke("What is his age ?")
print(res.content)

#### Implement history...

In [ ]:
history = []

In [ ]:
while True:
    query = input("User: ")
    if query == "exit":
        break
    
    history.append(HumanMessage(query))
    res = llm.invoke(history)
    history.append(AIMessage(content=res.content))
    print("AI: ", res.content)

In [ ]:
history